In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 
import pandas as pd
import ipywidgets
import ipywidgets as widgets
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d
import os

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size']= 15
plt.style.use("seaborn-colorblind")

# Functions

In [ ]:
def load_file(directory):
    """Reads rheometer file to extract data of interest based on experiment type."""
    target_variables = ["Time","Relaxation Modulus","Shear Stress","Strain"] #check stress relaxation files!
    skiprows=2
    with open(directory, mode='r', encoding='utf-8', errors='ignore') as f:
        #columns of interest to read and extract 
        reading=False
        magicheader = False 
        databox = []
        data = []
        names=[]
        for riga in f:
            if riga.startswith("Name:"):
                names.append(riga.strip()[8::]) #names of each test (curve)
            if reading is False:
                if riga.strip() == "Measuring Profile:": #start reading when we are at "Measuring Profile:"
                    reading=True
                    if len(data)>0:
                        databox.append(data) #np.array(data)
                    data = []
                    if magicheader is False: #we are at the header
                        for _ in range(skiprows): 
                            #skipping 3 lines
                            f.readline()
                        header = f.readline().strip().split('\t')
                        #storing index of variables of interest
                        headerIndex = []
                        for word in target_variables:
                            headerIndex.append(header.index(word))
                        magicheader = True
                        jump = 1 #jump 1 line if we are at header
                    else:
                        jump = 5 #jump 5 lines if we are at "Measuring Profile:"
                    for _ in range(jump):
                        #skipping lines
                        f.readline()
            else:
                if riga.strip() == "Data Series Information": 
                    reading=False
                    continue
                else:
                    if riga.strip()!='':
                        rigaall = np.array((riga.strip().replace(',', '.').split('\t')))
                        rigadata = rigaall[headerIndex] #takes only indexes from selected header variables
                        newline = list(map(float,rigadata)) #list
                        data.append(newline)
        if len(data)>0:
            databox.append(data) #np.array(data)
    return databox

def getMedCurve(xar, yar,loose=True, threshold=3, error=False):
    """
    Takes repeated nummerical data (replicates stored in a multi dimensional list) 
    and computes the average and error. Useful for displaying "average" plots
    with error bands.
    This function was taken from the following github repo (https://github.com/CellMechLab/nanoindentation),
    author Dr Massimo Vassalli at the Cellular Mechanobiology Lab, University of Glasgow.
    """
    if loose is False:
        xmin = -np.inf
        xmax = np.inf
        deltax = 0
        nonecount = 0
        for x in xar:
            if x is not None and np.min(x) is not None:
                xmin = np.max([xmin, np.min(x)])
                xmax = np.min([xmax, np.max(x)])
                deltax += ((np.max(x)-np.min(x))/(len(x)-1))
            else:
                nonecount += 1
        deltax /= (len(xar)-nonecount)
        xnew = np.linspace(xmin, xmax, int((xmax-xmin)/(deltax)))
        ynew = np.zeros(len(xnew))
        for i in range(len(xar)):
            if xar[i] is not None and np.min(xar[i]) is not None:
                ycur = np.interp(xnew, xar[i], yar[i])
                ynew += ycur
        ynew /= (len(xar)-nonecount)
    else:
        xmin = np.inf
        xmax = -np.inf
        deltax = 0
        for x in xar:
            try:
                xmin = np.min([xmin, np.min(x)])
                xmax = np.max([xmax, np.max(x)])
                deltax += ((np.max(x) - np.min(x)) / (len(x) - 1))
            except TypeError:
                return
        deltax /= len(xar)
        xnewall = np.linspace(xmin, xmax, int((xmax - xmin) / deltax))
        ynewall = np.zeros(len(xnewall))
        count = np.zeros(len(xnewall))
        ys = np.zeros([len(xnewall), len(xar)])
        for i in range(len(xar)):
            imin = np.argmin((xnewall - np.min(xar[i])) ** 2)  # +1
            imax = np.argmin((xnewall - np.max(xar[i])) ** 2)  # -1
            ycur = np.interp(xnewall[imin:imax], xar[i], yar[i])
            ynewall[imin:imax] += ycur
            count[imin:imax] += 1
            for j in range(imin, imax):
                ys[j][i] = ycur[j-imin]
        cc = count >= threshold
        xnew = xnewall[cc]
        ynew = ynewall[cc] / count[cc]
        yerrs_new = ys[cc]
        yerr = []
        for j in range(len(yerrs_new)):
            squr_sum = 0
            num = 0
            std = 0
            for i in range(0, len(yerrs_new[j])):
                if yerrs_new[j][i] != 0:
                    squr_sum += (yerrs_new[j][i] - ynew[j]) ** 2
                    num += 1
            if num > 0:
                std = np.sqrt(squr_sum / num)
            yerr.append(std)
        yerr = np.asarray(yerr)
    if error == False:
        return xnew[:-1], ynew[:-1]
    elif error == True:
        return xnew[:-1], ynew[:-1], yerr[:-1]

dirw=widgets.Text(
    value='',
    placeholder='Please enter the files directory',
    description='Directory:',
    disabled=False
)


Enter directory of file containing the stress relaxation data

In [ ]:
display(dirw)

Select the time window for plotting the data

In [ ]:
data=load_file(dirw.value)
def plot_data(tmin=0.0,tmax=1500.0):
    fig,ax=plt.subplots(1,2,figsize=(10,5))
    for i in range(len(data)): 
        #strain v time
        stress = np.array(data[i])[:,2]
        strain = np.array(data[i])[:,3]
        
        ax[0].loglog(np.array(data[i])[:,0],strain,alpha=0.5)
        ax[0].axvline(tmin,c="r",lw=1)
        ax[0].axvline(tmax,c="r",lw=1)
        
        ax[0].set_xlabel("t (s)")
        ax[0].set_ylabel(r"$\varepsilon$ (%)")


        #stress v time
        ax[1].loglog(np.array(data[i])[:,0], stress)
        ax[1].axvline(tmin,c="r",lw=1)
        ax[1].axvline(tmax,c="r",lw=1)
        
        ax[1].set_xlabel("t (s)")
        ax[1].set_ylabel(r"$\tau$ (Pa)")
        fig.tight_layout()
        
plot1w=widgets.interactive(plot_data,tmin=(0.0,50.0,0.5), tmax=(0.0,1500.0,0.5))
display(plot1w)

Plot normalised stress vs time for each curve together with the average curve. NB: average is computed over normalised data.

In [ ]:
#Widget parameters (Global thresholds)
t_min=plot1w.kwargs["tmin"] #s #time under which max force should occur
t_max=plot1w.kwargs["tmax"] #s #max time to display and analyse data for


fig,ax = plt.subplots(1,1,figsize=(7,5))
tall = []
stressall = []

for i in range(len(data)): 
    
    #Read data
    t=np.array(data[i])[:,0]
    stress=np.array(data[i])[:,2]
    strain = np.array(data[i])[:,3]

    #Slice data based on user-selected thresholds, and align time to 0
    itmin=np.argmin((t-t_min)**2)
    itmax=np.argmin((t-t_max)**2)
    t=t[itmin:itmax] - t[itmin]
    stress=stress[itmin:itmax] 
    
    #Filter stress signal and normalise to maximum stress
    stress = savgol_filter(stress,25,3)
    stress = stress/max(stress)
    
    #Plot individual curves
    plt.plot(t,stress,color='k',lw=0.5,alpha=0.5,label='test'+" "+str(i+1))
    
    #Append data for average curve calculation
    tall.append(t)
    stressall.append(stress)

tav = np.average(tall,axis=0)
stressav=np.average(stressall,axis=0)
stresssd = np.std(stressall,axis=0)

up = stressav + 0.5 * stresssd 
down = stressav -0.5 * stresssd

ax.plot(tav, stressav, color='b',lw=1,label='average')
ax.fill_between(tav,up,down, alpha=0.5,color='b',label='1SD')
plt.legend()

ax.set_xlabel("Time (s)")
ax.set_ylabel("Normalised Stress")

plt.show()
fig.tight_layout()

Save average data in an excel file in a directory of preference; then plot exported data in other software of preference for comparison

In [ ]:
#Please enter directory in the quotes
samplenamew=widgets.Text(
    placeholder='Please enter the sample name',
    disabled=False
)
namebox=widgets.HBox([widgets.Label(value="Sample Name:"), samplenamew])
display(namebox)

savedir=widgets.Text(
    placeholder='Please enter saving directory',
    disabled=False
)
savebox=widgets.HBox([widgets.Label(value="Saving directory:"), savedir])
display(savebox)

undersamplew = widgets.IntSlider(
    value=1,
    min=1,
    max=100,
    step=1,
    disabled=False,
    continuous_update=True,
    orientation='horizontal',
)
undersamplebox=widgets.HBox([widgets.Label(value="Undersample by:"), undersamplew])
display(undersamplebox)

In [ ]:
sample_name=samplenamew.value
sample_name = sample_name + ".csv"
data={"Normalised stress":stressav[0::undersamplew.value], "Standard deviation":stresssd[0::undersamplew.value]/2.0, 
      "Time (s)": tav[0::undersamplew.value]}
df=pd.DataFrame(data)
df.to_csv(os.path.join(savedir.value,sample_name),index=False)